# DIRAC in Jupyter Interface 
This notebook is a combination of the tutorial notebooks at [DIRAC's Documentation](https://gitlab.com/dirac/dirac/-/blob/master/demo_notebooks/). Note that since this is an optional part, there are extra dependencies to run these notebooks: 
1. ASE: [Atomistic Simulation Environment](https://ase-lib.org/). Installed with `pip install git+https://gitlab.com/ase/ase.git@master`.
2. DIRAC: [DIRAC](https://gitlab.com/dirac/dirac/-/blob/master) must be compiled and added to the path for the pam command (DIRAC's calculation handler) to be availiable to perform DIRAC calciulations.   
3. ase_dirac: located in the [DIRAC repository](https://gitlab.com/dirac/dirac/-/blob/master/) under the `ase_dirac` folder. Installed with `pip install -e $DIRAC_PATH/ase_dirac/`. 
4. h5py: necessary to extract information from DIRAC's `.h5` files generated in calculations.  

With those installed, the following notebook should be possible to be run. 

In [1]:
from ase import Atoms
from ase.build import molecule
from ase.units import Ha
from ase_dirac import DIRAC
from typing import Dict, List

## ASETutorial.ipynb
This is the content of the [first tutorial notebook](https://gitlab.com/dirac/dirac/-/blob/master/demo_notebooks/ASETutorial.ipynb) in the demo notebooks. It shows how to run a simple SCF and 

In [2]:
molecule_label = "HCl"
ase_molecule = molecule(molecule_label)

ase_molecule.calc = DIRAC(
    wave_function={
        ".scf": "",
        "*scf": {".ergcnv": "1.E-8 1.E-6", ".maxitr": "35"},
    },
    molecule={"*basis": {".default": "cc-pVDZ"}},
    label=molecule_label,
)
print(
    f"The Hartree-Fock energy of {molecule_label} is ",
    ase_molecule.get_potential_energy() / Ha,
    " Hartree.",
)

 starting DIRAC for molecule ./HCl.xyz with input ./HCl.inp
  creating archive file  HCl.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal
The Hartree-Fock energy of HCl is  -460.08944528048306  Hartree.


In [3]:
# Some simple small molecules taken from the G2 set in the ASE database
molecules = ["H2O", "N2", "F2"]
# Defines the **HAMILTONIAN input block, note the two lines of input for the full DCG Hamiltonian
hamiltonian_inputs = [".nonrel", ".x2cmmf", ".gaunt\n.dossss"]
# Labeling the different inputs with a descriptive string
calculation_labels = ["NR-MP2", "X2Cmmf-MP2", "DCG-MP2"]
# Making an empty list to store results (energies) for each molecule
results = []

# Outermost loop over different molecules
for mol in molecules:
    ase_molecule = molecule(mol)
    energy = []
    # Loop over the three different Hamiltonians that we consider
    for ham, calc in zip(hamiltonian_inputs, calculation_labels):
        label = calc + "_" + mol
        try:
            ase_molecule.calc = DIRAC(
                hamiltonian={ham: ""},
                wave_function={
                    ".scf": "",
                    #".mp2": "",
                    "*scf": {".ergcnv": "1.E-8 1.E-6", ".maxitr": "35"},
                    #"*mp2cal": {".occup": "all", ".virtual": "all"},
                    "**RELCCSD": {'.energy' : ''}
                },
                molecule={"*basis": {".default": "cc-pVDZ"}},
                label=label,
            )
            energy.append(ase_molecule.get_potential_energy() / Ha)
        except:
            print(f"Calculation {label} could not be run correctly, please check")
            energy.append(0)
    results.append(energy)

# Print results in a table
print(
    " Møller-Plesset second order (MP2) energies computed with different Hamiltonians\n"
)
print(80 * "-")
print("  {:<15} {:>20} {:>20} {:>20}".format("Molecule", *calculation_labels))
print(80 * "-")
for mol, result in zip(molecules, results):
    print("  {:<15} {:>20.8f} {:>20.8f} {:>20.8f}".format(mol, *result))
print(80 * "-")

 starting DIRAC for molecule ./NR-MP2_H2O.xyz with input ./NR-MP2_H2O.inp
  creating archive file  NR-MP2_H2O.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal
 starting DIRAC for molecule ./X2Cmmf-MP2_H2O.xyz with input ./X2Cmmf-MP2_H2O.inp
  creating archive file  X2Cmmf-MP2_H2O.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'X2CMAT', 'dirac.xml']
  exit           : normal
 starting DIRAC for molecule ./DCG-MP2_H2O.xyz with input ./DCG-MP2_H2O.inp
  creating archive file  DCG-MP2_H2O.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal
 starting DIRAC for molecule ./NR-MP2_N2.xyz with input ./NR-MP2_N2.inp
  creating archive file  NR-MP2_N2.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal
 starting DIRAC for molecule ./X2Cmmf-MP2_N2.xyz w

Which is actually very convenient and nice to perform calculations. Lets try something a little bit more complicated. We will try to translate a FS-DIP calculation, run it and extract the relevant results. Considering the input: 

```
**DIRAC
.TITLE
Molecular Te. DEA from the dianion.
.WAVE F
**HAMILTONIAN
.LVCORR
**WAVE FUNCTIONS
.SCF
.RELCCSD
*SCF
.CLOSED SHELL
 54 52
**RELCCSD
.ENERGY
.PRINT
1
.FOCKSP
*CCFSPC
.DOIE2
.NACTH
 3 3 2 2
.MAXIT
200
**MOLECULE
*BASIS
.DEFAULT
dyall.3zp
*END OF
```

In [4]:
fscc_str = """ **DIRAC
    .TITLE
    Molecular Te. DEA from the dianion.
    .WAVE F
    **HAMILTONIAN
    .LVCORR
    **WAVE FUNCTIONS
    .SCF
    .RELCCSD
    ! because we want this
    *SCF
    .PRINT 
    1
    ! .CLOSED SHELL
    ! 54 52
    **RELCCSD
    .ENERGY
    .PRINT
    1
    .FOCKSP
    *CCFSPC
    .DOIE2
    .NACTH
    0 0 1 1
    .MAXIT
    200
    **Moltra
    .Active
    energy -15.0 20.0 1.0
    **MOLECULE
    *BASIS
    .DEFAULT
    dyall.3zp
    *END OF
    """

# A minimal parser/constructor
Here is a minimal dirac parser/constructor to run calculations with ase_dirac but using predefined input files. Consider there is a `calc.inp`, this function will parse it, and add the possibility to build as:

```python
DIRAC.from_string(input_str: str)
```

Or:

```python
DIRAC.from_file(filename: str)
```

In [5]:
class DIRACC(DIRAC):

    @staticmethod
    def _parse_dirac_inp(inp_str: str) -> Dict:
        """Internal helper to parse raw DIRAC text into a nested dictionary."""
        accepted_keywords: List[str] = [
            "dirac", "general", "molecule", "hamiltonian",
            "wave_function", "analyze", "properties",
            "visual", "integrals", "grid", "moltra",
        ]

        # Clean input and add empty lines where required 
        inp_list = [
            line.strip() for line in inp_str.split("\n")
            if line.strip() != "" and not line.strip().startswith("!")
        ]

        to_insert = []
        for index in range(len(inp_list) - 1):
            line = inp_list[index]
            if line[0] == ".":
                next_line = inp_list[index + 1]
                if next_line[:2] == "**" or next_line[:1] in ["*", "."]:
                    to_insert.append(index + 1)

        for index in reversed(to_insert):
            inp_list.insert(index, "")

        inp_list.append('')


        # Locate the different input sections 
        double_stars = []
        single_stars = []
        points = []

        for index, line in enumerate(inp_list):
            if line[:2] == "**":
                double_stars.append(index)
            elif line[:1] == "*":
                single_stars.append(index)
            elif line[:1] == ".":
                points.append(index)

        # Extract the ** sections
        if len(double_stars) == 1:
            double_stars_slices = [slice(double_stars[0], len(inp_list))]
        elif len(double_stars) > 1:
            double_stars_slices = [
                slice(double_stars[i], double_stars[i + 1])
                for i in range(len(double_stars) - 1)
            ]
            double_stars_slices.append(slice(double_stars_slices[-1].stop, len(inp_list)))
        else:
            return {}

        # initialize dictionary
        input_dict = {}
        prev_double = None

        # Indicate type of each input line
        keyword_type = ["R" for _ in range(len(inp_list))]
        for index in double_stars: keyword_type[index] = "D"
        for index in single_stars: keyword_type[index] = "S"
        for index in points:       keyword_type[index] = "P"

        # Parse by ** section 
        for dslice in double_stars_slices:
            slice_list = inp_list[dslice]
            type_list = keyword_type[dslice]
            
            # read ** section keyword and initialize dictionary 
            inp_keyword = slice_list[0].lower()
            inp_keyword = "**wave function" if inp_keyword == "**wave functions" else inp_keyword # due to the keyword in the calculator
            dict_keyword = inp_keyword[2:].replace(" ", "_")

            # This overcomes cases where hierarchy is  ** -> ** -> . like **WAVE FUNCTION -> **RELCCSD -> .ENERGY
            if dict_keyword in accepted_keywords:
                input_dict.update({dict_keyword: {}})
                prev_double = dict_keyword
            else:
                type_list[0] = "S"

            # parent is defined to keep track of the ** -> * -> . hierarchy
            parent = None

            # Examine the hierarchy of the content of each ** slice
            for index, ttype in enumerate(type_list):
                # Define the ** keyword as parent of * 
                if ttype == "S":
                    parent = slice_list[index]
                    input_dict[prev_double].update({parent: {}})

                # The logic is that after any P, there must be at least an R due to construction
                if ttype == "P":
                    start_p_index = index
                    end_p_index = len(type_list) # Assume longest possible slice
                    
                    # Check if the actual slice is smaller 
                    for inndex, tttype in enumerate(type_list[index + 1 :]):
                        if tttype != "R":
                            end_p_index = start_p_index + inndex + 1
                            break

                    # once found, define slice, and content of P
                    p_slice = slice(start_p_index, end_p_index)
                    val_str = "\n".join(slice_list[p_slice][1:])

                    # Use parent to determine if the P dictionary should be added as ** -> . directly or as ** -> * -> .
                    # Within this logic, it is implicit that any ** that is not in the accepted keywords of the DIRAC class 
                    # will behave as ** -> ** -> . instead of trying to access a dictionary key that is not atmitted 
                    if parent is None:
                        input_dict[prev_double].update({slice_list[p_slice][0]: val_str})
                    else:
                        input_dict[prev_double][parent].update({slice_list[p_slice][0]: val_str})

        return input_dict

    @classmethod
    def from_string(cls, inp_str: str) -> "DIRAC":
        """
        DIRAC constructor from text string.
        """
        parsed_kwargs = cls._parse_dirac_inp(inp_str)
        # cls(**parsed_kwargs) is functionally identical to DIRAC(**parsed_kwargs)
        return cls(**parsed_kwargs)

    @classmethod
    def from_file(cls, filename: str) -> "DIRAC":
        """
        DIRAC constructor from  an input file.
        """
        with open(filename, "r") as f:
            inp_text = f.read()
        
        # We can call the other classmethod directly to avoid repeating logic
        return cls.from_string(inp_text)


In [6]:
from typing import List, Dict


def parse_dirac_inp(inp_str: str) -> Dict:
    accepted_keywords: List[str] = [
        "dirac",
        "general",
        "molecule",
        "hamiltonian",
        "wave_function",
        "analyze",
        "properties",
        "visual",
        "integrals",
        "grid",
        "moltra",
    ]

    inp_list = inp_str.split("\n")

    inp_list = [line.strip() for line in inp_list]

    to_insert = []
    for index in range(len(inp_list) - 1):
        line = inp_list[index]
        
        if line != "" and line[0] == ".":
            next_line = inp_list[index + 1]

            if next_line[:2] == "**" or next_line[:1] in ["*", "."]:
                to_insert.append(index + 1) 
                

    for index in reversed(to_insert):
        inp_list.insert(index, "")
    
    inp_list.append('')

    double_stars = []
    single_stars = []
    points = []

    for index, line in enumerate(inp_list):
        # print(line[:2])
        if line[:2] == "**":
            double_stars.append(index)
        elif line[:1] == "*":
            single_stars.append(index)
        elif line[:1] == ".":
            points.append(index)

    double_stars_slices = []

    if len(double_stars) == 1:
            double_stars_slices = [slice(double_stars[0], len(inp_list))]

    elif len(double_stars) > 1:
        double_stars_slices = [
            slice(double_stars[i], double_stars[i + 1])
            for i in range(len(double_stars) - 1)
        ]
        double_stars_slices.append(slice(double_stars_slices[-1].stop, len(inp_list)))

    input_dict = {}
    prev_double = None

    keyword_type = ["R" for i in range(len(inp_list))]

    for index in double_stars:
        keyword_type[index] = "D"

    for index in single_stars:
        keyword_type[index] = "S"

    for index in points:
        keyword_type[index] = "P"

    for dslice in double_stars_slices:
        slice_list = inp_list[dslice]
        type_list = keyword_type[dslice]
        # print(slice_list)
        inp_keyword = slice_list[0].lower()
        inp_keyword = (
            "**wave function" if inp_keyword == "**wave functions" else inp_keyword
        )

        dict_keyword = inp_keyword[2:].replace(" ", "_")

        if dict_keyword in accepted_keywords:
            input_dict.update({dict_keyword: {}})
            prev_double = dict_keyword

        else:
            type_list[0] = "S"

        parent = None

        for index, ttype in enumerate(type_list):
            if ttype == "S":
                parent = slice_list[index]
                input_dict[prev_double].update({parent: {}})

            if ttype == "P":
                start_p_index = index
                end_p_index = len(type_list)
                for inndex, tttype in enumerate(type_list[index + 1 :]):
                    if tttype != "R":
                        end_p_index = start_p_index + inndex + 1
                        break

                # print(start_p_index, end_p_index)

                p_slice = slice(start_p_index, end_p_index)
                # print(type_list[p_slice])

                if parent is None:
                    input_dict[prev_double].update(
                        {slice_list[p_slice][0]: "\n".join(slice_list[p_slice][1:])}
                    )

                elif parent is not None:
                    input_dict[prev_double][parent].update(
                        {slice_list[p_slice][0]: "\n".join(slice_list[p_slice][1:])}
                    )

    return input_dict


def dirac_calc_from_text(inp_text: str) -> DIRAC:
    return DIRAC(**parse_dirac_inp(inp_text))


def dirac_calc_from_input_file(filename: str) -> DIRAC:
    with open(filename, "r") as f:
        inp_text = f.read()
    return DIRAC(**parse_dirac_inp(inp_text))
